# Alpha -> таблица «Alpha Июль 2026»


Формат АТС здесь — РЕЕСТР ЗВОНКОВ (по строке на звонок),
в отличие от проектов «Мониторинга», где АТС отдаёт готовые почасовые суммы.

Что уходит на лист с датой:
    A1      реестр CRM с формулой округления минут в колонке G
    L3      сводка по часам: все / принятые / пропущенные / исходящие
    L36     разбивка принятых по операторам + строка Итого
    A<...>  реестр АТС под CRM-блоком

Порядок работы в DataSpell:
    1. Ячейка 1 — настройки
    2. Ячейка 2 — расчёт, проверка цифр на экране
    3. Ячейка 3 — выгрузка


## ЯЧЕЙКА 1 — НАСТРОЙКИ

In [ ]:
import pandas as pd

from etl_core import (
    DATA_DIR, CREDENTIALS, env,
    DateResolver, ExcelReader, MultiDay, SheetsLoader, Serializer,
    Validator, Palette, preview, log, month_title, HOURS, TIME_LABELS,
)

BASE_DIR = str(DATA_DIR)   # выгрузки лежат в папке data/ в корне репозитория
# CREDENTIALS импортируется из etl_core: credentials.json в корне, в git не попадает
# etl_core обновляется вместе с ноутбуками, а Python держит уже
# загруженный модуль в памяти: заменить файл мало, нужен Restart Kernel.
# Проверяем сразу, иначе это выяснится на середине выгрузки.
_need = ["req_number_format", "req_merge", "open_sheet_for_date"]
_missing = [m for m in _need if not hasattr(SheetsLoader, m)]
if _missing:
    raise ImportError(
        "etl_core.py устарел — в нём нет методов: " + ", ".join(_missing)
        + "\n\n1. Положите свежий etl_core.py в папку etl/"
        + "\n2. Нажмите Restart Kernel — без этого Python продолжит "
          "использовать старую версию из памяти"
    )

# Таблица заводится помесячно: 1 августа появится
# «Alpha Август 2026» с другим ID. Поэтому таблица ищется
# ПО НАЗВАНИЮ, а месяц подставляется из даты файла — правка
# кода при смене месяца не нужна.
SPREADSHEET_TITLE = "Alpha {month} {year}"

# Запасной путь: чтобы работать с конкретной таблицей по ID,
# впишите его сюда и поставьте USE_TITLE = False
USE_TITLE = True
SPREADSHEET_ID = env("ALPHA_SHEET_ID")

CRM_FILE = "alpha_crm.xlsx"
PBX_FILE = "alpha_pbx.xls"

# Соответствие внутреннего номера линии и оператора.
# Матчинг идёт по номеру, а не по имени: имена приходят с разным
# регистром (Operator1 / operator3), а номер стабилен.
OPERATOR_LINES = {
    "Внутр. 101": "Оператор 1",
    "Внутр. 102": "Оператор 2",
    "Внутр. 103": "Оператор 3",
}

# Колонки без источника в АТС — выводятся пустыми.
# Оператор 4 в выгрузках не встречается, но колонка нужна в таблице.
EMPTY_OPERATOR_COLUMNS = ["Оператор 4"]

# Результаты АТС — ровно три значения
RES_ACCEPTED = "Обработан первым оператором"
RES_LOST = "Потерян"
RES_IVR = "Обработан IVR"        # исключается из отчёта

# Выгрузка CRM теряет первую запись периода. Файл берётся с запасом
# в один день назад: самый ранний день считается буферным, показывается
# на экране, но не грузится — именно у него откушена первая запись.
USE_BUFFER_DAY = True

reader = ExcelReader(BASE_DIR)

## ЯЧЕЙКА 2 — РАСЧЁТ

In [ ]:
def build(crm: pd.DataFrame, inf: pd.DataFrame, d) -> dict:
    """
    Считает все блоки отчёта за ОДИН день.

    :param crm: срез CRM за этот день (может быть пустым — выходной)
    :param inf: срез реестра АТС за этот день
    :param d: дата отчёта

    Возвращает словарь с датафреймами: crm, inf (реестр АТС), hourly,
    operators, checks и датой.
    """
    crm = crm.copy()
    crm["_dt"] = pd.to_datetime(crm["Время начала звонка"], errors="coerce")

    # В выходные операторы не работают, и выгрузка CRM приходит пустой.
    # Это не ошибка: звонки в АТС всё равно есть (потерянные и IVR),
    # и отчёт за такой день должен строиться.
    crm_empty = not crm["_dt"].notna().any()
    if crm_empty:
        log.warning("%s: CRM пустой — исходящие приму за 0",
                    DateResolver.sheet_name(d))

    inf = inf.copy()
    inf["_h"] = pd.to_datetime(
        inf["Время"].astype(str), format="%H:%M:%S", errors="coerce"
    ).dt.hour

    unparsed = int(inf["_h"].isna().sum())
    if unparsed:
        log.warning("АТС: %d строк с нераспознанным временем", unparsed)
    inf = inf.dropna(subset=["_h"])
    inf["_h"] = inf["_h"].astype(int)

    # Полный реестр сохраняем как есть — он выгружается на лист целиком,
    # вместе со строками «Обработан IVR».
    inf_full = inf.copy()

    # А из ПОДСЧЁТОВ (почасовая сводка, операторы) IVR исключается:
    # в отчёте нужны только входящие, пропущенные и исходящие.
    ivr_count = int((inf["Результат"] == RES_IVR).sum())
    inf = inf[inf["Результат"] != RES_IVR].copy()
    log.info("IVR-звонков: %d — в реестре остаются, в расчёты не входят",
             ivr_count)

    # --- почасовая сводка ---
    accepted = inf[inf["Результат"] == RES_ACCEPTED]
    lost = inf[inf["Результат"] == RES_LOST]

    if crm_empty:
        out_mask = pd.Series(False, index=crm.index)
        outbound = pd.Series(0, index=HOURS)
    else:
        out_mask = crm["Направление звонка"].astype(str).str.contains(
            "Исход", case=False, na=False
        )
        outbound = (crm[out_mask].dropna(subset=["_dt"])
                    .groupby(crm.loc[out_mask, "_dt"].dt.hour).size()
                    .reindex(HOURS, fill_value=0))

    hourly = pd.DataFrame({
        d.strftime("%d.%m.%Y"): [TIME_LABELS[h] for h in HOURS],
        "Принятые входящие вызовы": accepted.groupby("_h").size().reindex(HOURS, fill_value=0).values,
        "Пропущенные вызовы": lost.groupby("_h").size().reindex(HOURS, fill_value=0).values,
        "Исходящие вызовы": outbound.values,
    })
    # «Все вызовы» — сумма трёх показателей, а не число строк в файле
    hourly.insert(1, "Все вызовы",
                  hourly["Принятые входящие вызовы"]
                  + hourly["Пропущенные вызовы"]
                  + hourly["Исходящие вызовы"])

    # Строка «Итого» — считается здесь, вручную формулы ставить не надо
    metric_cols = ["Все вызовы", "Принятые входящие вызовы",
                   "Пропущенные вызовы", "Исходящие вызовы"]
    total = {d.strftime("%d.%m.%Y"): "Итого"}
    total.update({c: int(hourly[c].sum()) for c in metric_cols})
    hourly = pd.concat([hourly, pd.DataFrame([total])], ignore_index=True)

    # --- разбивка принятых по операторам ---
    operators = pd.DataFrame({d.strftime("%d.%m.%Y"): [TIME_LABELS[h] for h in HOURS]})
    for line, label in OPERATOR_LINES.items():
        cnt = (accepted[accepted["Внутренний"] == line]
               .groupby("_h").size().reindex(HOURS, fill_value=0))
        operators[label] = cnt.values

    op_cols = list(OPERATOR_LINES.values())

    total_row = {d.strftime("%d.%m.%Y"): "Итого"}
    total_row.update({c: int(operators[c].sum()) for c in op_cols})
    operators = pd.concat([operators, pd.DataFrame([total_row])], ignore_index=True)

    # Пустые колонки добавляются после расчёта итогов, чтобы не попасть
    # в сверку: считать по ним нечего
    for c in EMPTY_OPERATOR_COLUMNS:
        operators[c] = ""

    # --- сверки ---
    # Итоговая строка уже добавлена, поэтому берём её значения напрямую
    h_tot = hourly.iloc[-1]

    checks = Validator.report([
        # Главная проверка: сумма по операторам обязана совпасть с входящими
        ("Сумма по операторам vs Входящие",
         int(operators.loc[operators.index[-1], op_cols].sum()),
         int(h_tot["Принятые входящие вызовы"])),
        ("Входящие: почасовая сумма vs файл АТС",
         int(h_tot["Принятые входящие вызовы"]), int(len(accepted))),
        ("Пропущенные: почасовая сумма vs файл АТС",
         int(h_tot["Пропущенные вызовы"]), int(len(lost))),
        ("Исходящие: почасовая сумма vs CRM",
         int(h_tot["Исходящие вызовы"]), int(out_mask.sum())),
        ("Все вызовы = Вход + Пропущ + Исход",
         int(h_tot["Все вызовы"]),
         int(h_tot["Принятые входящие вызовы"] + h_tot["Пропущенные вызовы"]
             + h_tot["Исходящие вызовы"])),
    ])

    # Операторы, которых нет в справочнике — иначе часть звонков потеряется молча
    known = set(OPERATOR_LINES)
    unknown = set(accepted["Внутренний"].unique()) - known
    if unknown:
        log.warning("Принятые звонки на линиях вне справочника: %s", unknown)

    return {"crm": crm, "inf": inf_full, "hourly": hourly,
            "operators": operators, "checks": checks, "date": d}


# --- чтение файлов и разбор по дням ---

crm_all = reader.read_crm(CRM_FILE)
inf_all = reader.read_pbx_registry(PBX_FILE)

crm_parts = MultiDay.split(crm_all, "Время начала звонка")
inf_parts = MultiDay.split(inf_all, "Дата", dayfirst=True)

# Дни берём по АТС: он есть всегда, даже когда CRM пустой
plan, DAYS_TO_LOAD, BUFFER_DAY = MultiDay.plan(inf_parts, USE_BUFFER_DAY)
preview(plan, f"ЧТО В ФАЙЛЕ {PBX_FILE}")

if BUFFER_DAY is not None:
    b = inf_parts[BUFFER_DAY]
    crm_in_buffer = len(crm_parts.get(BUFFER_DAY, []))
    print(f"\nБуферный день {BUFFER_DAY.strftime('%d.%m.%Y')} — "
          f"{len(b)} строк АТС"
          f"{f', {crm_in_buffer} строк CRM' if crm_in_buffer else ', CRM пуст'}"
          f". Грузиться НЕ будет.")

    # Буфер защищает, только если в нём ЕСТЬ строки CRM.
    #
    # CRM теряет первую запись запрошенного периода. Обычно это
    # первый звонок буферного дня — он и так выбрасывается, потери
    # нет. Но если операторы в буферный день не работали (воскресенье,
    # праздник), строк CRM за него ноль, и первой записью периода
    # оказывается первый звонок РАБОЧЕГО дня. Теряется именно он,
    # молча — АТС этот звонок видит, а CRM уже нет.
    #
    # Сам ноутбук починить это не может: записи просто нет в файле.
    # Поэтому предупреждаем, чтобы её добрали через reconciliation.ipynb.
    if not crm_in_buffer:
        print()
        print("   ВНИМАНИЕ: в буферном дне НЕТ строк CRM — операторы "
              "в этот день не работали.")
        print("   Значит первой записью периода стал первый звонок "
              "рабочего дня, и CRM его потеряла.")
        print(f"   День {DAYS_TO_LOAD[0].strftime('%d.%m.%Y') if DAYS_TO_LOAD else ''} "
              f"скорее всего пришёл без своего первого звонка.")
        print("   Что делать: качнуть период шире (DAYS в crm_reports.py) "
              "или добрать звонок через reconciliation.ipynb.")
    preview(b.sort_values("Время").head(5)[
        ["Дата", "Время", "Результат", "Внутренний", "Оператор"]],
        f"БУФЕРНЫЙ ДЕНЬ {BUFFER_DAY.strftime('%d.%m.%Y')} — НЕ ГРУЗИТСЯ")

if not DAYS_TO_LOAD:
    raise ValueError("Грузить нечего: в файле только буферный день. "
                     "Возьмите период шире.")

# --- расчёт по каждому дню ---

EMPTY_CRM = crm_all.iloc[0:0]
results = {}
for d in DAYS_TO_LOAD:
    results[d] = build(crm_parts.get(d, EMPTY_CRM), inf_parts[d], d)

print(f"\nПосчитано дней: {len(results)}")
for d in DAYS_TO_LOAD:
    h = results[d]["hourly"].iloc[-1]
    print(f"   {DateResolver.sheet_name(d)}: все {int(h['Все вызовы'])}, "
          f"входящие {int(h['Принятые входящие вызовы'])}, "
          f"пропущенные {int(h['Пропущенные вызовы'])}, "
          f"исходящие {int(h['Исходящие вызовы'])}")

# Сверки по всем дням разом: одна не сошлась — не грузим ничего
all_checks = pd.concat(
    [results[d]["checks"].assign(День=DateResolver.sheet_name(d))
     for d in DAYS_TO_LOAD], ignore_index=True)
preview(all_checks, "СВЕРКА ПО ВСЕМ ДНЯМ")

bad = all_checks[all_checks["OK"] == "✗"]
assert bad.empty, f"Расхождение:\n{bad.to_string()}"

## ЯЧЕЙКА 3 — ВЫГРУЗКА

In [ ]:
def upload(data: dict) -> None:
    """Выгружает все блоки на лист с датой и красит по образцу."""
    d = data["date"]
    crm, inf = data["crm"], data["inf"]
    hourly, operators = data["hourly"], data["operators"]

    if USE_TITLE:
        loader = SheetsLoader(CREDENTIALS).connect(
            title=month_title(SPREADSHEET_TITLE, d))
    else:
        loader = SheetsLoader(CREDENTIALS, SPREADSHEET_ID).connect()

    loader.open_sheet_for_date(d, rows=1200, cols=30)

    # --- блок CRM (A1) с формулой округления минут ---
    headers_crm = [
        "ID звонка", "Направление звонка", "Номер телефона",
        "Время начала звонка", "Время конца звонка", "Продолжительность, сек",
        "Продолжительность, минутах с учётом округления", "Оператор",
        "Тема звонка / вопросы", "Тема №4",
    ]
    # None — формула, "" — пустая колонка под ручное заполнение.
    # «Тема №4» остаётся пустой: все темы звонка уже перечислены
    # в колонке I, дублировать четвёртую отдельно не нужно.
    src_cols = ["ID звонка", "Направление звонка", "Номер телефона",
                "Время начала звонка", "Время конца звонка",
                "Продолжительность, сек", None, "Оператор",
                "Тема звонка / вопросы", ""]

    rows_crm = []
    for i, (_, row) in enumerate(crm.iterrows()):
        r = i + 2                       # строка в Sheets
        line = []
        for c in src_cols:
            if c is None:
                line.append(f"=CEILING(F{r}/60)")
            elif c == "":
                line.append("")
            else:
                line.append(Serializer.cast(row.get(c)))
        rows_crm.append(line)

    # --- блок АТС под CRM ---
    # Заголовок-полоса на строке (inf_start - 2), шапка на (inf_start - 1),
    # данные с inf_start.
    inf_start = len(rows_crm) + 5
    # Подписи идут в том же порядке, что и колонки ниже: сначала линия,
    # на которую пришёл звонок, потом номер абонента. Раньше эти две
    # подписи стояли наоборот, и в колонке «Номер» лежал номер линии —
    # один и тот же у всех строк. Фильтровать по клиенту было нельзя.
    headers_inf = ["Дата", "Время", "Результат", "Номер линии", "Номер",
                   "IVR", "Имя оператора", "Продолжительность", "Итого"]
    inf_out = inf[["Дата", "Время", "Результат", "Линия", "Номер",
                   "IVR_длит", "Оператор", "Длит_оператора", "Итого"]]
    rows_inf = Serializer.frame(inf_out, header=False)

    payload = [
        {"range": "A1", "values": [headers_crm] + rows_crm},
        {"range": f"A{inf_start - 2}",
         "values": [[f"Продолжительность IVR {d.strftime('%d.%m.%Y')}"]]},
        {"range": f"A{inf_start - 1}", "values": [headers_inf]},
        {"range": f"A{inf_start}", "values": rows_inf},
        {"range": "L3", "values": Serializer.frame(hourly, header=True)},
        {"range": "L36", "values": Serializer.frame(operators, header=True)},
    ]
    loader.write(payload)

    # --- форматирование ---
    # Индексы строк нулевые: строка N в Sheets -> N-1 здесь
    band_row = inf_start - 3        # заголовок-полоса
    head_row = inf_start - 2        # шапка блока АТС
    n_inf = len(rows_inf)

    reqs = [
        loader.req_header(0, 0, 10, Palette.HEADER_GREEN),
        loader.req_freeze(1),
        loader.req_borders(0, len(rows_crm) + 1, 0, 10),

        # Заголовок-полоса: объединён на всю ширину блока, тёмная заливка
        loader.req_merge(band_row, band_row + 1, 0, 9),
        loader.req_header(band_row, 0, 9, Palette.HEADER_DARK,
                          font_color=Palette.WHITE),

        # Шапка блока АТС
        loader.req_header(head_row, 0, 9, Palette.HEADER_DARK,
                          font_color=Palette.WHITE),
        loader.req_borders(band_row, head_row + 1 + n_inf, 0, 9),

        # сводный блок L3 — тёмная шапка, белый текст
        # 1 шапка + 24 часа + 1 Итого = 26 строк
        loader.req_header(2, 11, 16, Palette.HEADER_DARK,
                          font_color=Palette.WHITE),
        loader.req_borders(2, 28, 11, 16),
        loader.req_center(3, 28, 11, 16),
        loader.req_row_fill(27, 11, 16, Palette.TOTAL_YELLOW, bold=True),

        # блок операторов L36 — тёмная шапка
        loader.req_header(35, 11, 11 + len(operators.columns), Palette.HEADER_DARK,
                          font_color=Palette.WHITE),
        loader.req_borders(35, 61, 11, 11 + len(operators.columns)),
        loader.req_center(36, 61, 11, 11 + len(operators.columns)),
        loader.req_row_fill(60, 11, 11 + len(operators.columns),
                            Palette.TOTAL_YELLOW, bold=True),

        loader.req_col_width(11, 12, 110),
        loader.req_col_width(12, 16, 100),

    ]

    # Подсветка активных часов: только там, где были принятые звонки.
    # Строка 18:00-19:00 с пропущенными, но без принятых — не красится,
    # как и в текущей таблице.
    for i, v in enumerate(hourly["Принятые входящие вызовы"]):
        if v > 0:
            reqs.append(loader.req_row_fill(3 + i, 11, 16, Palette.ROW_YELLOW))
            reqs.append(loader.req_row_fill(36 + i, 11,
                                            11 + len(operators.columns),
                                            Palette.ROW_YELLOW))

    # Умные таблицы не создаются: метод addTable нестабилен.
    # Заливка, рамки и подсветка применяются обычным форматированием,
    # а фильтры при необходимости включаются вручную:
    # выделить блок -> Формат -> Преобразовать в таблицу.
    loader.apply_format(reqs, strict=False)

    print(f"\nГотово: лист '{DateResolver.sheet_name(d)}' в «{loader.sh.title}»")


def upload_all() -> None:
    """Заливает каждый посчитанный день на свой лист."""
    for d in DAYS_TO_LOAD:
        upload(results[d])
    if BUFFER_DAY is not None:
        print(f"\nБуферный день {BUFFER_DAY.strftime('%d.%m.%Y')} "
              f"не загружался.")


upload_all()